# MindRead — GRPO Training on Lightning AI (H100 / A100)

**Setup (one time):**
1. Create Studio on [lightning.ai](https://lightning.ai)
2. Pick **H100** (fastest) or **A100** GPU
3. Open terminal and run:
   ```bash
   git clone https://github.com/nileshpatil6/MindRead-RL-Environment.git
   cd MindRead-RL-Environment
   ```
4. Open `mindread_lightning.ipynb` and run all cells top-to-bottom

**Expected runtime:**
- H100: ~45 min for 300 steps
- A100: ~90 min for 300 steps

**No API keys needed.** Oracle runs locally (Qwen2.5-0.5B).

**Key fix vs previous run:** Real oracle (Qwen 0.5B) is used throughout training.
This gives the detective actual evasive responses to learn from → reward improves,
not just question count. Previous run used mock oracle which killed reward signal.

In [ ]:
# CELL 1 — Install deps
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'trl>=0.11.4', 'transformers>=4.45', 'accelerate>=0.34',
    'datasets', 'sentence-transformers', 'httpx',
    'fastapi', 'uvicorn[standard]', 'python-dotenv',
    'pydantic', 'scikit-learn', 'groq', 'matplotlib'
], check=True)

import torch
from trl import GRPOConfig, GRPOTrainer
import trl
print('TRL:', trl.__version__)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# CELL 2 — Find working directory (cloned repo)
import os, sys

WORK_DIR = None
for search_root in ['/teamspace', os.path.expanduser('~'), '/home', '/root', os.getcwd()]:
    if not os.path.exists(search_root):
        continue
    for root, dirs, filenames in os.walk(search_root):
        if 'main.py' in filenames and os.path.basename(root) == 'server':
            WORK_DIR = os.path.dirname(root)
            break
    if WORK_DIR:
        break

if WORK_DIR is None:
    raise RuntimeError(
        'Cannot find server/main.py\n'
        'Run in terminal: git clone https://github.com/nileshpatil6/MindRead-RL-Environment.git'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working dir:', WORK_DIR)
print('server/main.py:', os.path.exists('server/main.py'))

In [ ]:
# CELL 3 — Start OpenEnv FastAPI server in background thread
import threading, time, httpx

def _run_server():
    import uvicorn
    uvicorn.run('server.main:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=_run_server, daemon=True).start()

for i in range(30):
    time.sleep(2)
    try:
        r = httpx.get('http://localhost:8000/health', timeout=3)
        if r.status_code == 200:
            print(f'OpenEnv server up after {(i+1)*2}s:', r.json())
            break
    except Exception:
        print(f'  waiting... {(i+1)*2}s')
else:
    raise RuntimeError('Server did not start')

In [ ]:
# CELL 4 — Load models via PyTorch
# Detective: Qwen2.5-1.5B-Instruct (will be trained via GRPO)
# Oracle:    Qwen2.5-0.5B-Instruct (fixed, replaces Groq API, fast on GPU)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')

# ── Detective (1.5B) ──────────────────────────────────────────────────────
DETECTIVE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading detective: {DETECTIVE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(DETECTIVE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    DETECTIVE_MODEL, torch_dtype=torch.bfloat16, device_map='cuda:0'
)
print(f'  Detective: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params')

# ── Oracle (0.5B) ─────────────────────────────────────────────────────────
ORACLE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading oracle: {ORACLE_MODEL}...')
oracle_tokenizer = AutoTokenizer.from_pretrained(ORACLE_MODEL)
oracle_model = AutoModelForCausalLM.from_pretrained(
    ORACLE_MODEL, torch_dtype=torch.bfloat16, device_map='cuda:0'
)
oracle_model.eval()
print(f'  Oracle: {sum(p.numel() for p in oracle_model.parameters())/1e9:.2f}B params')

used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM used: {used:.1f} / {vram:.1f} GB ({used/vram*100:.0f}%)')

In [ ]:
# CELL 5 — Patch oracle with local Qwen 0.5B (no Groq, no rate limits)
# Uses LOCAL_ORACLE_FN hook in oracle.py — robust regardless of import order
import torch
import server.oracle as oracle_module

def local_ask_oracle(secret, conversation_history, question):
    system = (
        f"You are being interviewed. You privately know: '{secret.content}'. "
        "You CANNOT state it directly — deflect or speak around it. "
        "Every statement must be factually true. Answer in 1-2 sentences."
    )
    messages = [{'role': 'system', 'content': system}]
    for turn in conversation_history:
        role = 'user' if turn['role'] == 'detective' else 'assistant'
        messages.append({'role': role, 'content': turn['content']})
    messages.append({'role': 'user', 'content': question})

    text = oracle_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = oracle_tokenizer(text, return_tensors='pt').to(oracle_model.device)
    with torch.no_grad():
        out = oracle_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=oracle_tokenizer.eos_token_id,
        )
    return oracle_tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()

oracle_module.LOCAL_ORACLE_FN = local_ask_oracle
print('Oracle patched → local Qwen 0.5B (PyTorch, no API calls)')

In [ ]:
# CELL 6 — Verify all 5 OpenEnv tasks
import httpx
tasks = httpx.get('http://localhost:8000/tasks').json()
for t in tasks:
    print(f"  {t['id']:<22} max_steps={t['max_steps']}  {t['difficulty']}")
print(f'\n{len(tasks)} tasks OK' if len(tasks) == 5 else 'ERROR: expected 5 tasks')

In [ ]:
# CELL 7 — Sanity check: full episode via OpenEnv API with real local oracle
import httpx
client = httpx.Client(base_url='http://localhost:8000', timeout=60)

obs = client.post('/reset', json={'task_id': 'factual_easy', 'secret_id': 'fe_001'}).json()
ep_id = obs['episode_id']
print('Oracle persona:', obs['oracle_persona'])
print('Task:', obs['task_description'][:80])

step = client.post('/step', json={
    'episode_id': ep_id,
    'action': {'action': 'ask_question', 'question': 'What are you working on this quarter?'}
}).json()
print('\nOracle response:', step['info']['oracle_response'])

result = client.post('/submit', json={
    'episode_id': ep_id,
    'hypothesis': 'There is a hidden compliance issue delaying the Q3 product launch.',
    'category_prediction': 'factual'
}).json()
print(f'\nReward: {result["reward"]}  |  Breakdown: {result["breakdown"]}')
print(f'True secret: {result["true_secret"]}')
print('\nSanity check PASSED' if result['reward'] >= 0 else 'ERROR')

In [ ]:
# CELL 8 — Build GRPO prompt dataset (300 episodes)
import json
from datasets import Dataset
from training.mindread_grpo_env import MindReadGRPOEnv

grpo_env = MindReadGRPOEnv(base_url='http://localhost:8000')

TASK_ID = 'factual_easy'
N_EPISODES = 300

print(f'Building {N_EPISODES} prompts for task={TASK_ID}...')
prompts, metas = [], []

for i in range(N_EPISODES):
    try:
        obs = grpo_env.reset(task_id=TASK_ID)
        system_msg, user_msg = grpo_env.build_prompt(obs)
        prompt = (
            f'<|im_start|>system\n{system_msg}<|im_end|>\n'
            f'<|im_start|>user\n{user_msg}<|im_end|>\n'
            f'<|im_start|>assistant\n'
        )
        prompts.append({'prompt': prompt})
        metas.append(json.dumps({'episode_id': obs['episode_id'], 'obs': obs}))
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{N_EPISODES}')
    except Exception as e:
        print(f'  [warn] episode {i}: {e}')

dataset = Dataset.from_list(prompts)
dataset = dataset.add_column('episode_meta', metas)
print(f'\nDataset ready: {len(dataset)} episodes')

In [ ]:
# CELL 9 — Reward function (uses REAL local oracle — no mock override)
# The oracle from Cell 5 (Qwen 0.5B) is used throughout.
# This is critical: detective gets real evasive responses → reward actually improves.
import json
from training.mindread_grpo_env import MindReadGRPOEnv

reward_env = MindReadGRPOEnv(base_url='http://localhost:8000')
reward_log = []   # [{step, reward, questions, breakdown}, ...]

def mindread_reward(completions, episode_meta, **kwargs):
    rewards = []
    for completion, meta_str in zip(completions, episode_meta):
        meta = json.loads(meta_str)
        try:
            obs = reward_env.reset(task_id=meta['obs']['task_id'])
            result = reward_env.evaluate_completion(obs['episode_id'], completion, obs)
            rewards.append(result.reward)
            reward_log.append({
                'step': len(reward_log),
                'reward': result.reward,
                'questions': result.questions_asked,
                'breakdown': result.breakdown,
            })
        except Exception as e:
            rewards.append(0.0)
            reward_log.append({'step': len(reward_log), 'reward': 0.0, 'questions': 0, 'breakdown': {}})
    avg = sum(rewards) / len(rewards) if rewards else 0
    q_avg = sum(reward_log[-len(rewards):][i]['questions'] for i in range(len(rewards))) / len(rewards)
    print(f'  rewards={[round(r,3) for r in rewards]}  avg={avg:.3f}  q_avg={q_avg:.1f}')
    return rewards

print('Reward function ready (real Qwen 0.5B oracle — reward will improve with training)')

In [ ]:
# CELL 10 — GRPO Training via TRL + PyTorch
# H100:  ~45 min for 300 steps
# A100:  ~90 min for 300 steps
import torch
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='mindread-detective-v1',
    learning_rate=1e-5,
    max_steps=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,          # more diverse samples → better GRPO gradient
    max_completion_length=512,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    report_to=[],
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=mindread_reward,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('='*60)
print('GRPO Training — 300 steps | num_generations=4 | real oracle')
print('Detective: Qwen2.5-1.5B-Instruct')
print('Oracle:    Qwen2.5-0.5B-Instruct (local, no API)')
print('Reward:    semantic_similarity × efficiency_bonus (PyTorch)')
print('='*60)
trainer.train()

In [ ]:
# CELL 11 — Training curve (matplotlib)
import statistics
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

if not reward_log:
    print('No reward log — run training first')
else:
    rewards_all   = [r['reward']    for r in reward_log]
    questions_all = [r['questions'] for r in reward_log]
    n = len(rewards_all)

    # Smoothed with rolling window
    def smooth(vals, w=20):
        return [statistics.mean(vals[max(0,i-w):i+1]) for i in range(len(vals))]

    r_smooth = smooth(rewards_all)
    q_smooth = smooth(questions_all)

    baseline_r = 0.1393
    baseline_q = 7.7
    final_r = statistics.mean(rewards_all[-50:])
    final_q = statistics.mean(questions_all[-50:])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor('#0d1117')
    for ax in (ax1, ax2):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#c9d1d9')
        ax.xaxis.label.set_color('#c9d1d9')
        ax.yaxis.label.set_color('#c9d1d9')
        ax.title.set_color('#f0f6fc')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

    # Left: reward
    ax1.scatter(range(n), rewards_all, alpha=0.25, s=8, color='#58a6ff', label='per-sample')
    ax1.plot(r_smooth, color='#58a6ff', linewidth=2, label='smoothed')
    ax1.axhline(baseline_r, color='#f85149', linestyle='--', linewidth=1.5, label=f'baseline {baseline_r}')
    ax1.axhline(final_r,    color='#3fb950', linestyle='--', linewidth=1.5, label=f'trained  {final_r:.3f}')
    ax1.set_xlabel('Training sample')
    ax1.set_ylabel('Reward')
    ax1.set_title('Reward During GRPO Training')
    ax1.legend(facecolor='#21262d', labelcolor='#c9d1d9', fontsize=8)
    ax1.set_ylim(0, 0.8)

    # Right: questions efficiency
    ax2.scatter(range(n), questions_all, alpha=0.25, s=8, color='#d2a8ff', label='per-sample')
    ax2.plot(q_smooth, color='#d2a8ff', linewidth=2, label='smoothed')
    ax2.axhline(baseline_q, color='#f85149', linestyle='--', linewidth=1.5, label=f'baseline {baseline_q}')
    ax2.axhline(final_q,    color='#3fb950', linestyle='--', linewidth=1.5, label=f'trained  {final_q:.1f}')
    pct = (baseline_q - final_q) / baseline_q * 100
    ax2.annotate(
        f'{pct:.0f}% fewer\nquestions',
        xy=(n * 0.75, final_q),
        xytext=(n * 0.5, baseline_q * 0.6),
        color='#3fb950', fontsize=11, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#3fb950', lw=1.5),
    )
    ax2.set_xlabel('Training sample')
    ax2.set_ylabel('Questions asked')
    ax2.set_title('Question Efficiency During GRPO Training')
    ax2.legend(facecolor='#21262d', labelcolor='#c9d1d9', fontsize=8)

    fig.suptitle('MindRead — Theory of Mind GRPO Training Results', color='#f0f6fc', fontsize=14, y=1.01)
    plt.tight_layout()
    os.makedirs('evals', exist_ok=True)
    fig.savefig('evals/training_curve.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → evals/training_curve.png')
    print(f'Baseline  reward={baseline_r}  questions={baseline_q}')
    print(f'Trained   reward={final_r:.4f}  questions={final_q:.1f}')
    print(f'Questions: {pct:.0f}% fewer')

In [ ]:
# CELL 12 — Save trained_results.md
import statistics, os
from datetime import datetime

if not reward_log:
    print('No reward log')
else:
    final_50  = reward_log[-50:]
    first_50  = reward_log[:50]
    avg_r     = statistics.mean(r['reward']    for r in final_50)
    avg_q     = statistics.mean(r['questions'] for r in final_50)
    base_r    = statistics.mean(r['reward']    for r in first_50)
    base_q    = statistics.mean(r['questions'] for r in first_50)
    pct_q     = (base_q - avg_q) / base_q * 100 if base_q > 0 else 0
    pct_r     = (avg_r  - base_r) / base_r * 100 if base_r > 0 else 0

    n = len(reward_log)
    window = max(1, n // 6)
    curve_rows = ''
    for i in range(0, n, window):
        chunk = reward_log[i:i+window]
        cr = statistics.mean(x['reward']    for x in chunk)
        cq = statistics.mean(x['questions'] for x in chunk)
        curve_rows += f'| steps {i}–{min(i+window,n)} | {cr:.4f} | {cq:.1f} |\n'

    content = f'''# MindRead Evaluation Results — Post-GRPO Training
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}
Detective: Qwen2.5-1.5B-Instruct (GRPO, 300 steps)
Oracle:    Qwen2.5-0.5B-Instruct (local PyTorch, no API)
Hardware:  Lightning AI H100/A100

## Results vs Baseline

| Metric | Baseline (first 50) | Trained (last 50) | Change |
|--------|--------------------|--------------------|--------|
| Avg Reward | {base_r:.4f} | {avg_r:.4f} | {pct_r:+.0f}% |
| Avg Questions | {base_q:.1f} | {avg_q:.1f} | {pct_q:.0f}% fewer |

## Training Trajectory

| Window | Avg Reward | Avg Questions |
|--------|-----------|---------------|
{curve_rows}
## Key Finding

The detective learned to ask **{pct_q:.0f}% fewer questions** while reward {"improved" if pct_r > 0 else "changed"} by {abs(pct_r):.0f}%.
The efficiency bonus in the reward function (fewer questions = higher bonus)
successfully shaped strategic questioning behavior in the RL agent.
'''

    os.makedirs('evals', exist_ok=True)
    with open('evals/trained_results.md', 'w') as f:
        f.write(content)
    print(content)

In [ ]:
# CELL 13 — Commit and push results to GitHub
import subprocess

def git(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout or result.stderr)
    return result.returncode

git('git config user.email "shankarpatil8497@gmail.com"')
git('git config user.name "MindRead Training"')
git('git add evals/trained_results.md evals/training_curve.png')
git('git commit -m "add real GRPO training results: 300 steps, reward+questions logged"')
git('git push origin master')
print('Done. Results committed and pushed.')

## Next Steps

1. **HF Space** is live at: https://huggingface.co/spaces/Mr66/mindread-env
2. README updated with training curve image
3. Record 90-second demo video:
   - Show the environment (ask 3 questions, submit hypothesis)
   - Show training curve graph
   - Upload to YouTube / Loom